In [21]:


df_gpt = pd.read_parquet('../experiments/dewi_upload/claude_4_5_predictions.parquet')
df_gpt = pd.read_parquet('../experiments/dewi_upload/gemini_3_predictions.parquet')
#df_gpt = pd.read_parquet('../experiments/dewi_upload/gemma_3_predictions.parquet')
df_gpt = pd.read_parquet('../experiments/dewi_upload/qwen_3_predictions.parquet')
df_gpt = pd.read_parquet('../experiments/dewi_upload/gpt_5_predictions.parquet')
#df_gpt = pd.read_parquet('../experiments/dewi_upload/claude_4_5_predictions.parquet')

df_gpt['without_acc'] = df_gpt['counterfactual_predictor_response_without_explanation_answer'] == df_gpt['counterfactual_reference_response_answer']
df_gpt['with_acc'] = df_gpt['counterfactual_predictor_response_with_explanation_answer'] == df_gpt['counterfactual_reference_response_answer']

def helped(row):
    if row['without_acc'] == row['with_acc']:
        return 0
    if (row['without_acc'] == True) and (row['with_acc'] == False):
        return -1
    if (row['without_acc'] == False) and (row['with_acc'] == True):
        return 1

df_gpt['explanation_helped'] = df_gpt.apply(helped, axis=1)

df_gpt.groupby('original_reference_response_model_info_model').apply(
    lambda g: g['original_reference_response_reasoning_tokens'].corr(g['explanation_helped'])
)

/tmp/ipykernel_1280356/1421475594.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_gpt.groupby('original_reference_response_model_info_model').apply(


original_reference_response_model_info_model
openai/gpt-5-mini   -0.062528
openai/gpt-5-nano   -0.052687
openai/gpt-5.2      -0.043332
dtype: float64

In [23]:
import duckdb
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.contingency_tables import mcnemar

In [25]:
files = [
    '../experiments/dewi_upload/claude_4_5_predictions.parquet',
    '../experiments/dewi_upload/gemini_3_predictions.parquet',
    '../experiments/dewi_upload/qwen_3_predictions.parquet',
    '../experiments/dewi_upload/gpt_5_predictions.parquet',
]

for file in files:
    print(f"\n{'='*60}")
    print(f"File: {file}")
    print('='*60)
    
    df_gpt = pd.read_parquet(file)
    
    # Drop rows where any of these columns are None
    cols_to_check = [
        'counterfactual_predictor_response_without_explanation_answer',
        'counterfactual_predictor_response_with_explanation_answer',
        'counterfactual_reference_response_answer',
        'original_reference_response_answer'
    ]
    df_gpt = df_gpt.dropna(subset=cols_to_check)
    
    df_gpt['without_acc'] = df_gpt['counterfactual_predictor_response_without_explanation_answer'] == df_gpt['counterfactual_reference_response_answer']
    df_gpt['with_acc'] = df_gpt['counterfactual_predictor_response_with_explanation_answer'] == df_gpt['counterfactual_reference_response_answer']

    def helped(row):
        if row['without_acc'] == row['with_acc']:
            return 0
        if (row['without_acc'] == True) and (row['with_acc'] == False):
            return -1
        if (row['without_acc'] == False) and (row['with_acc'] == True):
            return 1

    df_gpt['explanation_helped'] = df_gpt.apply(helped, axis=1)

    print(df_gpt.groupby('original_reference_response_model_info_model').apply(
        lambda g: g['original_reference_response_reasoning_tokens'].corr(g['explanation_helped'])
    ))



File: ../experiments/dewi_upload/claude_4_5_predictions.parquet


/tmp/ipykernel_1280356/557660605.py:37: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(df_gpt.groupby('original_reference_response_model_info_model').apply(


original_reference_response_model_info_model
anthropic/claude-haiku-4.5    -0.034920
anthropic/claude-opus-4.5     -0.017549
anthropic/claude-sonnet-4.5   -0.051854
dtype: float64

File: ../experiments/dewi_upload/gemini_3_predictions.parquet


/tmp/ipykernel_1280356/557660605.py:37: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(df_gpt.groupby('original_reference_response_model_info_model').apply(


original_reference_response_model_info_model
google/gemini-3-flash-preview   -0.057441
google/gemini-3-pro-preview     -0.019891
dtype: float64

File: ../experiments/dewi_upload/qwen_3_predictions.parquet


/tmp/ipykernel_1280356/557660605.py:37: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(df_gpt.groupby('original_reference_response_model_info_model').apply(


original_reference_response_model_info_model
Qwen/Qwen3-0.6B   NaN
Qwen/Qwen3-1.7B   NaN
Qwen/Qwen3-14B    NaN
Qwen/Qwen3-32B    NaN
Qwen/Qwen3-4B     NaN
Qwen/Qwen3-8B     NaN
dtype: float64

File: ../experiments/dewi_upload/gpt_5_predictions.parquet
original_reference_response_model_info_model
openai/gpt-5-mini   -0.061177
openai/gpt-5-nano   -0.039887
openai/gpt-5.2      -0.042011
dtype: float64


/tmp/ipykernel_1280356/557660605.py:37: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(df_gpt.groupby('original_reference_response_model_info_model').apply(


In [26]:
df_gpt

,original_dataset,original_question,original_question_prompt,original_question_idx,original_ground_truth,original_answer_first,original_description,original_question_options,original_reference_response_cot,original_reference_response_raw_response,...,counterfactual_predictor_response_without_explanation_predictor_answers,counterfactual_predictor_response_without_explanation_predictor_names,counterfactual_predictor_response_without_explanation_input_tokens,counterfactual_predictor_response_without_explanation_reasoning_tokens,counterfactual_predictor_response_without_explanation_output_tokens,counterfactual_predictor_counterfactual_testability_score,counterfactual_predictor_counterfactual_testability_cot,without_acc,with_acc,explanation_helped
0,moral_machines,Consider that pedestrians are crossing a zebra...,You will be presented with scenarios involving...,13742,None,True,Consider that pedestrians are crossing a zebra...,None,None,[ANSWER]\nCASE 2\n\n[EXPLANATION]\nBoth cases ...,...,"[CASE 2, CASE 2, CASE 2, CASE 2, CASE 2]","[openai/gpt-oss-20b, google/gemma-3-27b-it, Qw...",None,None,None,8.0,We need to evaluate counterfactual testability...,True,True,0
1,heart_disease,"This is a male patient, experiencing non-angin...",You are a medical diagnosis assistant. Based o...,97,YES,False,"This is a male patient, experiencing non-angin...",None,None,[EXPLANATION]\nThis patient’s overall profile ...,...,"[YES, YES, YES, NO, YES]","[openai/gpt-oss-20b, google/gemma-3-27b-it, Qw...",None,None,None,7.0,We need to evaluate counterfactual testability...,True,True,0
2,heart_disease,"This is a male patient, experiencing atypical ...",You are a medical diagnosis assistant. Based o...,258,NO,True,"This is a male patient, experiencing atypical ...",None,None,"[ANSWER]\nNO\n\n[EXPLANATION]\nOverall, this p...",...,"[NO, YES, YES, NO, YES]","[openai/gpt-oss-20b, google/gemma-3-27b-it, Qw...",None,None,None,7.0,We need to evaluate counterfactual testability...,True,False,-1
3,attrition,"This is a female employee aged 18-30, married,...",You are analyzing employee data from a company...,122,NO,False,"This is a female employee aged 18-30, married,...",None,**Considering attrition prediction**\n\nI’m an...,[EXPLANATION]\nThis employee shows a mix of re...,...,"[NO, NO, YES, NO, NO]","[openai/gpt-oss-20b, google/gemma-3-27b-it, Qw...",None,None,None,5.0,We need to read the rubric and evaluate counte...,True,True,0
4,pima_diabetes,This is a woman of Southern Native American (P...,You are a medical assessment assistant special...,249,YES,True,This is a woman of Southern Native American (P...,None,**Predicting diabetes risk**\n\nI need to dete...,[ANSWER] \nNO\n\n[EXPLANATION] \nThis patien...,...,"[NO, NO, NO, NO, NO]","[openai/gpt-oss-20b, google/gemma-3-27b-it, Qw...",None,None,None,9.0,We need to evaluate counterfactual testability...,True,True,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20148,income,This is a Asian-Pac-Islander Male between 25 a...,You are analyzing 1994 census data from the Un...,7849,NO,False,This is a Asian-Pac-Islander Male between 25 a...,None,"**Analyzing income data**\n\nThe ""Census Incom...","[EXPLANATION]\nIn 1994 census data, a person w...",...,"[YES, YES, YES, YES, YES]","[openai/gpt-oss-20b, google/gemma-3-27b-it, Qw...",None,None,None,5.0,We need to evaluate counterfactual testability...,False,False,0
20149,bank_marketing,This is a married manager who is 50-59 years o...,You are a bank marketing analyst reviewing the...,18025,NO SUBSCRIPTION,True,This is a married manager who is 50-59 years o...,None,**Considering subscription probability**\n\nI'...,[ANSWER]\nSUBSCRIBED\n\n[EXPLANATION]\nThe app...,...,"[NO SUBSCRIPTION, SUBSCRIBED, NO SUBSCRIPTION,...","[openai/gpt-oss-20b, google/gemma-3-27b-it, Qw...",None,None,None,7.0,We need to evaluate counterfactual testability...,False,True,1
20150,bank_marketing,This is a single admin worker who is 30-39 yea...,You are a bank 